# Chapitre 3 - Lab : les index de PostgreSQL vus de l'intérieur

Ce lab confronte le cours à un vrai moteur : PostgreSQL 16, qui range ses tables dans un
**fichier de tas** et ses index dans des **B-trees** (cours 3 et 4). Nous allons :

1. mesurer un parcours séquentiel et lire un plan d'exécution (`EXPLAIN ANALYZE`) ;
2. inspecter la structure réelle d'un index B-tree (extension `pageinspect`) et observer
   le choix du plan selon la sélectivité ;
3. obtenir un **index-only scan** grâce à un index couvrant (`INCLUDE`) ;
4. mesurer le coût des index en écriture, en octets de **WAL** ;
5. observer la **fragmentation** après des suppressions et le rôle de `VACUUM` ;
6. construire un index **GIN** pour la recherche plein texte.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/storage/compose.yml -p ddia_storage up -d     # démarrer
docker compose -f docker/storage/compose.yml -p ddia_storage down -v   # arrêter et effacer
```

PostgreSQL écoute sur le port **10432** de la machine (utilisateur, mot de passe et base :
`ddia` / `ddia` / `stockage`).

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg

from ddia import storage as S
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)

DSN = "host=localhost port=10432 user=ddia password=ddia dbname=stockage"
con = None
for essai in range(30):
    try:
        con = psycopg.connect(DSN, autocommit=True)
        break
    except psycopg.OperationalError:
        time.sleep(1)
assert con is not None, "PostgreSQL ne répond pas : avez-vous lancé docker compose up ?"
print(con.execute("SELECT version()").fetchone()[0][:60])


def q(sql, params=None):
    """Exécute une requête et renvoie toutes les lignes."""
    return con.execute(sql, params).fetchall()


def plan(sql):
    """Plan réel (EXPLAIN ANALYZE, BUFFERS) sous forme de texte."""
    return "\n".join(r[0] for r in q("EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY OFF) " + sql))


def plan_json(sql):
    return q("EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) " + sql)[0][0][0]

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linu


## 1. Une table et un parcours séquentiel (section *Data Structures That Power Your Database*)

On crée une table de commandes d'un million de lignes, générée côté serveur
(`generate_series`) avec une graine fixe. PostgreSQL range les lignes dans un **tas** :
des pages de 8 Kio, dans l'ordre d'insertion, sans index pour l'instant.

In [2]:
con.execute("DROP TABLE IF EXISTS commandes")
con.execute("""
CREATE TABLE commandes (
    id        bigint,
    client    int,
    statut    text,
    jour      int,
    montant   numeric(10, 2),
    adresse   text
)""")
con.execute("SELECT setseed(0.42)")
con.execute("""
INSERT INTO commandes
SELECT g,
       (100000 * random() ^ 2)::int,                       -- clients : quelques gros clients
       (ARRAY['livrée','livrée','livrée','expédiée','payée','annulée'])[1 + (random() * 5)::int],
       (random() * 364)::int,
       round((random() * 200)::numeric, 2),
       repeat('x', 80)
FROM generate_series(1, 1000000) AS g""")
con.execute("VACUUM ANALYZE commandes")
taille = q("SELECT pg_relation_size('commandes'), pg_size_pretty(pg_relation_size('commandes'))")[0]
print(f"table : {taille[1]}, soit {taille[0] // 8192} pages de 8 Kio")
print(plan("SELECT sum(montant) FROM commandes WHERE client = 4242"))

table : 137 MB, soit 17555 pages de 8 Kio
Finalize Aggregate  (cost=23763.58..23763.59 rows=1 width=32) (actual rows=1 loops=1)
  Buffers: shared hit=17555
  ->  Gather  (cost=23763.35..23763.56 rows=2 width=32) (actual rows=3 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=17555
        ->  Partial Aggregate  (cost=22763.35..22763.36 rows=1 width=32) (actual rows=1 loops=3)
              Buffers: shared hit=17555
              ->  Parallel Seq Scan on commandes  (cost=0.00..22763.33 rows=7 width=6) (actual rows=10 loops=3)
                    Filter: (client = 4242)
                    Rows Removed by Filter: 333323
                    Buffers: shared hit=17555
Planning:
  Buffers: shared hit=29 read=2


Le plan montre un `Seq Scan` (parcours séquentiel, ici en parallèle) : PostgreSQL lit
**toutes** les pages du tas (`Buffers: shared hit=... read=...` : pages trouvées dans
son cache ou lues) pour trouver une poignée de lignes. C'est notre `db_get` du cours 1,
en plus rapide.

> **À retenir.** Sans index, toute requête sélective coûte un parcours de la table
> entière : le nombre de pages lues est la taille de la table.